Viewed pooling.ipynb:1-400
Viewed pooling.ipynb:401-526

这篇 Notebook [pooling.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_convolutional-neural-networks/pooling.ipynb) 主要介绍了卷积神经网络中的**汇聚层（Pooling Layer，常译为池化层）**。核心内容总结如下：

---

### 一、引入汇聚层的背景与作用

1. **逐渐聚合全局信息（扩大感受野）**：
   - 图像分类等宏观任务需要对整张图像敏感。通过逐层汇聚降低空间分辨率，使高层神经元拥有更大的感受野，从而学习到全局特征。
2. **平移不变性（降低位置敏感度）**：
   - 现实场景中边缘或目标物体常因震动、角度变化等发生 1~2 个像素的微小位移。汇聚层可以在一定范围内容忍这种位置偏移，保持特征表达的一致性。
3. **无学习参数**：
   - 与卷积层不同，汇聚层只执行确定性的窗口规约操作（如取最大值或均值），本身不包含任何可训练参数。

---

### 二、最大汇聚层与平均汇聚层

在输入特征图上滑动固定形状（如 `p × q`）的汇聚窗口：
* **最大汇聚（Max Pooling）**：取窗口内所有元素的最大值，更偏向保留最显著的边缘或纹理特征。
* **平均汇聚（Average Pooling）**：取窗口内所有元素的平均值，更偏向平滑特征。

#### 从零实现（`pool2d` 前向传播逻辑）
```python
def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i: i + p_h, j: j + p_w].max()
            elif mode == 'avg':
                Y[i, j] = X[i: i + p_h, j: j + p_w].mean()
    return Y
```

---

### 三、填充（Padding）与步幅（Stride）

Notebook 演示了如何使用 PyTorch 内置的 `nn.MaxPool2d`：

1. **默认步幅行为**：
   * 在深度学习框架（如 PyTorch）中，汇聚层的**步幅默认等于汇聚窗口大小**。例如 `nn.MaxPool2d(3)` 的汇聚窗口是 `(3, 3)`，默认步幅也是 `(3, 3)`。
2. **自定义步幅与填充**：
   * 可以显式传入 `padding` 和 `stride` 参数来精准控制下采样的输出尺寸，如：
     ```python
     # 3x3 窗口，padding 为 1，步幅为 2
     pool2d = nn.MaxPool2d(3, padding=1, stride=2)
     
     # 自定义非对称矩形窗口及步幅
     pool2d = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
     ```

---

### 四、多通道运算特性

* **与卷积层的核心区别**：
  * **卷积层**：多通道输入会与对应的卷积核做互相关后再在通道维度上求和汇总（改变通道数）。
  * **汇聚层**：**在每个输入通道上各自独立运算**，不跨通道合并。
* **输入输出通道数一致**：输入如果是 `c × h × w`，汇聚后尺寸为 `c × h' × w'`，通道数 `c` 保持不变。

---

### 五、核心要点回顾

| 对比维度 | 卷积层（Conv2d） | 汇聚层（MaxPool2d / AvgPool2d） |
| :--- | :--- | :--- |
| **可学习参数** | 有（权重 `W` 和偏置 `b`） | **无**（固定确定性运算） |
| **通道处理方式** | 跨通道加权求和，通道数由卷积核个数决定 | **各通道独立计算**，输出通道数恒等于输入通道数 |
| **主要功能** | 特征提取与特征组合 | 空间降采样、扩大感受野、提供平移不变性 |

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [3]:
def pool_2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = torch.zeros(size=(X.shape[0]-p_h+1, X.shape[1]-p_w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i:i+p_h, j:j+p_w].max()
            if mode == 'avg':
                Y[i, j] = X[i:i+p_h, j:j+p_w].mean()
    return Y

In [8]:
X = torch.arange(9, dtype=torch.float32).reshape((3,3))
pool_2d(X, (2,2))

tensor([[4., 5.],
        [7., 8.]])

In [9]:
pool_2d(X, (2,2), 'avg')

tensor([[2., 3.],
        [5., 6.]])

In [10]:
X = torch.arange(16, dtype=torch.float32).reshape(1,1,4,4)


In [11]:
pool2d = nn.MaxPool2d(3)

In [12]:
pool2d(X)

tensor([[[[10.]]]])

In [13]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])

In [15]:
pool2d = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]]]])

In [14]:
X = torch.cat((X, X+1), 1)
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]],

         [[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])

In [16]:
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]]]])

以下是针对 [pooling.ipynb](file:///Users/songchao/WorkSpace/Python/DeepLearn/pytorch/chapter_convolutional-neural-networks/pooling.ipynb) 课后练习题的详细解答与代码分析：

---

### 练习 1：尝试将平均汇聚层作为卷积层的特殊情况实现

#### 思路与原理
* **平均汇聚（Average Pooling）**：窗口大小为 `p_h × p_w` 时，输出即为窗口内所有输入像素的算术平均值。
* **等价为卷积**：
  1. 卷积核尺寸设为 `(p_h, p_w)`；
  2. 卷积核的所有权重固定为 `1 / (p_h · p_w)`，偏置设为 0；
  3. 因为汇聚层在各通道间是独立计算的，所以卷积层必须使用**分组卷积**（即设置 `groups = channels`，实现 Depthwise 卷积）；
  4. 权重固定，不需要计算梯度（`requires_grad = False`）。

#### 代码实现与验证
```python
import torch
from torch import nn

def avg_pool2d_by_conv(X, pool_size, stride=None, padding=0):
    if isinstance(pool_size, int):
        p_h = p_w = pool_size
    else:
        p_h, p_w = pool_size
    
    stride = pool_size if stride is None else stride
    c = X.shape[1]
    
    # 构造分组卷积层
    conv = nn.Conv2d(in_channels=c, out_channels=c, 
                     kernel_size=(p_h, p_w), stride=stride, 
                     padding=padding, groups=c, bias=False)
    
    # 将每个通道的卷积核权重初始化为常数 1 / (p_h * p_w)
    weight_val = 1.0 / (p_h * p_w)
    conv.weight.data.fill_(weight_val)
    conv.weight.requires_grad = False
    
    return conv(X)

# 验证与 nn.AvgPool2d 的一致性
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
out_conv = avg_pool2d_by_conv(X, pool_size=2, stride=2)
out_pool = nn.AvgPool2d(kernel_size=2, stride=2)(X)

print("卷积模拟平均汇聚结果:\n", out_conv)
print("两者是否等价:", torch.allclose(out_conv, out_pool))
```

---

### 练习 2：尝试将最大汇聚层作为卷积层的特殊情况实现

#### 思路与结论
* **单层标准线性卷积无法直接实现最大汇聚**：
  * 标准卷积属于**线性变换**（加权求和加上偏置）；
  * 最大汇聚本质上是**高度非线性**的操作（从多个候选中选出极值）。对于任意输入，线性加权无法动态模拟出 `max(a, b)`。
* **如何通过广义神经网络 / 多层结构逼近/实现**：
  1. **利用 ReLU 构造比较器（多层卷积网络）**：
     数学上有恒等式：
     ```text
     max(a, b) = 0.5 · (a + b + |a - b|)
               = 0.5 · (a + b + ReLU(a - b) + ReLU(b - a))
     ```
     通过级联多个包含特定权重和 ReLU 激活函数的小卷积层，构建二叉树形式的两两比较，理论上可以实现多输入的最大值挑选。
  2. **形态学卷积（Max-Plus 卷积 / 热带几何）**：
     在数学形态学中，若将卷积定义中的“乘法”换成“加法”、“求和”换成“取最大值”，即得到极值代数下的卷积，但这已超出了 PyTorch 标准线性卷积 `nn.Conv2d` 的范畴。

---

### 练习 3：输入大小为 c × h × w，汇聚窗口 p_h × p_w，填充 (pad_h, pad_w)，步幅 (s_h, s_w)，计算成本是多少？

#### 1. 输出特征图尺寸
对于单通道输出的高度与宽度：
```text
h_out = ⌊(h - p_h + 2 · pad_h) / s_h⌋ + 1
w_out = ⌊(w - p_w + 2 · pad_w) / s_w⌋ + 1
```
总输出元素个数为：`c × h_out × w_out`。

#### 2. 每个输出元素的计算成本
对于单个汇聚窗口（共 `p_h × p_w` 个输入元素）：
* **平均汇聚（Avg Pooling）**：
  * 加法次数：`(p_h · p_w - 1)` 次加法；
  * 除法/乘法次数：`1` 次（除以窗口大小或乘以倒数）；
  * 单个窗口计算成本约为 `p_h · p_w` 次浮点运算（FLOPs）。
* **最大汇聚（Max Pooling）**：
  * 比较次数：`(p_h · p_w - 1)` 次大小比较运算。

#### 3. 总计算成本
```text
总计算次数 ≈ c × h_out × w_out × (p_h · p_w)
```
> **对比分析**：
> 标准卷积层不仅需要跨通道乘加运算，且包含参数乘法，其计算量约为 `c_in × c_out × h_out × w_out × (p_h · p_w)`。汇聚层既无乘法也无跨通道聚合，计算成本比标准卷积低几个数量级。

---

### 练习 4：为什么最大汇聚层和平均汇聚层的工作方式不同？

二者的统计学意义与保留的信息类型不同：

1. **最大汇聚（Max Pooling）**：
   * **机制**：仅保留局部区域中响应最强（最大）的信号，抑制其他微弱特征和背景噪声。
   * **特点**：对边缘、纹理、亮点等具有很高的选择性；具备更强的平移不变性（目标只要在窗口内，最强响应就能被传递到下一层）。
   * **常见位置**：广泛用于现代 CNN 的浅层和中层特征下采样（如 VGG、ResNet 的骨干部分）。

2. **平均汇聚（Average Pooling）**：
   * **机制**：取整个区域的均值，对全区域的所有像素赋予同等重视。
   * **特点**：平滑特征图，保留低频背景信息，但会“稀释”局部极其突出的强响应特征。
   * **常见位置**：常用于网络的最后一层（即**全局平均汇聚 Global Average Pooling，GAP**），将整张特征图压缩成 1 个数值，用于替代参数量庞大的全连接层。

---

### 练习 5：是否需要最小汇聚层？可以用已知函数替换它吗？

#### 1. 是否需要最小汇聚层？
* **常规 CNN 中很少需要**：因为现代 CNN 普遍使用 ReLU 作为激活函数，未激活的神经元（或背景）输出均为 0，有效特征响应均为正数。若使用最小汇聚，窗口极易取到 0，导致大量有效特征被清零淹没。
* **特定场景有需求**：如检测黑色缺陷/暗部斑点、距离场计算、或负向特征提取时，最小汇聚仍具有其实用价值。

#### 2. 用已知函数替换最小汇聚
完全可以利用数学恒等式：
```text
min(X) = -max(-X)
```
即通过现有的 `MaxPool2d` 配合取负操作实现最小汇聚：
```python
def min_pool2d(X, pool_size, stride=None, padding=0):
    max_pool = nn.MaxPool2d(pool_size, stride=stride, padding=padding)
    return -max_pool(-X)
```

---

### 练习 6：除了平均汇聚和最大汇聚，是否有其它函数可以考虑？为什么它不流行？

#### 1. 其他可选的汇聚函数
1. **平滑最大汇聚（Soft Pooling / LogSumExp Pooling）**：
   受 Softmax 启发，对窗口内的元素按照大小分配权重后求加权和：
   ```text
   y = ∑ (x_i · exp(β · x_i)) / ∑ exp(β · x_i)
   ```
   * 当参数 `β → 0` 时，退化为平均汇聚；
   * 当参数 `β → +∞` 时，逼近最大汇聚。
2. **p 范数汇聚（p-norm / L2 Pooling）**：
   计算窗口向量的范数 `(∑ |x_i|^p )^(1/p)`，也是介于平均和最大之间的一种泛化。
3. **随机汇聚（Stochastic Pooling）**：
   将窗口内激活值归一化为概率分布，按照概率随机采样其中的一个值作为输出，相当于一种数据增强/正则化手段。

#### 2. 为什么它们不流行？
1. **计算复杂度高**：`exp`、对数、高次幂以及除法运算的硬件计算开销远高于简单的“求和/比较”指令，拖慢训练与推理速度。
2. **数值稳定性差**：指数运算在深度网络中极易发生上溢（Overflow）或下溢（Underflow）。
3. **边际效益低**：实验证明，Soft Pooling 等复杂汇聚带来的精度提升非常有限，无法弥补其增加的计算延迟。
4. **被步幅卷积替代**：现代主流架构（如 ResNet、ConvNeXt）更倾向于直接使用**步幅大于 1 的卷积（Strided Convolution）**完成下采样，让网络自主学习最优的下采样滤波器，逐渐减少了对手工设计复杂池化函数的依赖。